# Machine Learning for Forecasting

Companion code for the *Predictive Analytics Lab* (section 27). Run the cell below.

In [ ]:
# Tree ensembles on lag features with recursive multi-step forecasting
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
AIR = [112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
y = np.array(AIR, dtype=float); tr, te = y[:120], y[120:]
L = 12

def lagged(s):
    return np.array([s[t - L:t] for t in range(L, len(s))]), s[L:]

def forecast(model, series, diff=False):
    s = np.diff(series) if diff else series
    X, Y = lagged(s); model.fit(X, Y)
    hist, level, out = list(s), series[-1], []
    for _ in range(24):
        v = model.predict([hist[-L:]])[0]; hist.append(v)
        level = level + v if diff else v; out.append(level)
    return np.array(out)

for name, m in [("Random Forest", RandomForestRegressor(100, random_state=0)), ("Gradient boosting", GradientBoostingRegressor(random_state=0))]:
    for diff in (False, True):
        f = forecast(m, tr, diff)
        print(f"{name:18s} {'differences' if diff else 'raw levels ':11s} RMSE={np.sqrt(np.mean((f - te) ** 2)):6.2f}  max forecast={f.max():.0f}")